# 02 — HRNet-W48 Validation

Notebook da US #13 / task #15 para validar o wrapper `HRNetEstimator` e comparar com a referência do paper.

**Objetivos:**
- Confirmar que o ONNX exportado produz keypoints válidos em imagens do 3DSP;
- Calcular PDJ@0.5 no split de validação;
- Documentar por que o HRNet performa ~56% (quantização de heatmaps em crops 100×100);
- Comparar com RTMPose lado a lado.

**Referência esperada (Yeung et al., 2024, Tab. 5):** PDJ@0.5 ≈ **56.08%**

## Dependências

O `HRNetEstimator` usa `onnxruntime` para inferência. Instale caso necessário:

In [ ]:
!uv pip install onnxruntime-gpu

## Setup

In [ ]:
import os
from pathlib import Path
import sys

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
if not DATA_DIR.is_absolute():
    DATA_DIR = ROOT / DATA_DIR

SPLIT_CONFIG = Path(os.getenv("SPLIT_CONFIG", "configs/split.json"))
if not SPLIT_CONFIG.is_absolute():
    SPLIT_CONFIG = ROOT / SPLIT_CONFIG

ONNX_PATH = ROOT / "models" / "weights" / "hrnet_w48_coco_256x192.onnx"

print("ROOT:        ", ROOT)
print("DATA_DIR:    ", DATA_DIR)
print("SPLIT_CONFIG:", SPLIT_CONFIG)
print("ONNX:        ", ONNX_PATH)
print("ONNX existe: ", ONNX_PATH.exists())

## Checagem da GPU

In [ ]:
!nvidia-smi

In [ ]:
import onnxruntime as ort

print("onnxruntime:", ort.__version__)

if hasattr(ort, "preload_dlls"):
    try:
        ort.preload_dlls()
        print("ort.preload_dlls(): ok")
    except Exception as exc:
        print("ort.preload_dlls(): falhou —", repr(exc))

providers = ort.get_available_providers()
print("providers:", providers)

DEVICE = "cuda" if "CUDAExecutionProvider" in providers else "cpu"
print(f"\nUsando device: {DEVICE}")

## Inferência em uma imagem do 3DSP

In [ ]:
import numpy as np

from football_orient_pose.estimators import HRNetEstimator
from football_orient_pose.utils.data_io import load_clip_image

estimator = HRNetEstimator(
    model_path=str(ONNX_PATH),
    device=DEVICE,
)

image = load_clip_image(DATA_DIR / "train" / "00001", frame_idx=1)

kp_coco = estimator.predict(image)
kp_h3wb = estimator.predict_h3wb(image)

print("COCO shape: ", kp_coco.shape)
print("H3WB shape: ", kp_h3wb.shape)
print("conf min/max:", float(kp_coco[:, 2].min()), float(kp_coco[:, 2].max()))

assert kp_coco.shape == (17, 3)
assert kp_h3wb.shape == (17, 2)

### Visualização dos keypoints na imagem

In [ ]:
import matplotlib.pyplot as plt
import cv2

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, (kp, title) in zip(axes, [
    (kp_coco, "COCO-17 (HRNet output)"),
    (kp_h3wb, "H3WB-17 (após conversão)"),
]):
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.scatter(kp[:, 0], kp[:, 1], s=20, c="red", zorder=5)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
plt.show()

## Smoke test em batch

In [ ]:
images = [load_clip_image(DATA_DIR / "train" / "00001", frame_idx=i) for i in range(1, 6)]
kp_batch = estimator.predict_batch(images)

print("Batch shape:", kp_batch.shape)
assert kp_batch.shape == (5, 17, 3)

## Validação PDJ no split de validação

Referência esperada: **PDJ@0.5 ≈ 56.08%** (Yeung et al., 2024, Tab. 5)

> **Por que o HRNet é ruim em crops 100×100?**  
> O HRNet gera heatmaps de resolução `(17, 64, 48)`. Em um crop 100×100,
> cada pixel do heatmap corresponde a ~2 px da imagem original.
> Esse erro de quantização se acumula em todos os joints, derrubando o PDJ
> para ~56% — muito abaixo dos 93% do RTMPose, que usa SimCC (coordenadas diretas).

In [ ]:
import json

from football_orient_pose.evaluation import compute_pdj
from football_orient_pose.utils.data_io import load_keypoints_2d

split_data = json.loads(SPLIT_CONFIG.read_text())
clip_ids = split_data["val"]   # 40 clips × 20 frames = 800 frames

predictions, targets = [], []

from tqdm.notebook import tqdm

for clip_id in tqdm(clip_ids, desc="Inferência HRNet"):
    clip_dir = DATA_DIR / "train" / clip_id
    for frame_idx in range(1, 21):
        img = load_clip_image(clip_dir, frame_idx)
        predictions.append(estimator.predict_h3wb(img))
        targets.append(load_keypoints_2d(clip_dir / "posture" / f"{frame_idx:03d}.json"))

predictions = np.asarray(predictions, dtype=np.float32)
targets     = np.asarray(targets,     dtype=np.float32)

pdj = compute_pdj(predictions, targets, threshold=0.5)

print(f"Frames válidos: {pdj.valid_frames}")
print(f"PDJ@0.5:        {pdj.global_score * 100:.2f}%")
print(f"Ref (paper):    56.08%")
print("\nPDJ por grupo:")
for group, score in pdj.per_group.items():
    print(f"  {group:<12}: {score * 100:.2f}%")

## Métricas complementares

In [ ]:
from football_orient_pose.evaluation import (
    compute_pck,
    compute_oks,
    compute_mpjpe_2d,
    joint_detection_report,
)

pck   = compute_pck(predictions, targets, threshold=0.2)
oks   = compute_oks(predictions, targets)
mpjpe = compute_mpjpe_2d(predictions, targets)
det   = joint_detection_report(predictions, targets, threshold=0.5)

print("="*55)
print(f"  {'Métrica':<20}  {'HRNet':>10}  {'RTMPose':>10}")
print("="*55)
print(f"  {'PDJ@0.5':<20}  {pdj.global_score*100:>9.2f}%  {'93.25%':>10}")
print(f"  {'PCK@0.2':<20}  {pck.global_score*100:>9.2f}%  {'41.39%':>10}")
print(f"  {'OKS':<20}  {oks.global_oks*100:>9.2f}%  {'82.04%':>10}")
print(f"  {'AP50':<20}  {oks.ap50*100:>9.2f}%  {'97.15%':>10}")
print(f"  {'mAP@[.5:.95]':<20}  {oks.ap*100:>9.2f}%  {'69.59%':>10}")
print(f"  {'MPJPE-2D':<20}  {mpjpe.global_mpjpe:>9.2f}px  {'4.67px':>10}")
print(f"  {'F1-macro':<20}  {det.f1_macro*100:>9.2f}%  {'93.25%':>10}")
print("="*55)

## Comparação por grupo: HRNet vs RTMPose

In [ ]:
# Resultados RTMPose (train split, notebook 01)
rtmpose_pdj = {"head": 99.45, "shoulder": 97.75, "elbow": 89.78,
               "wrist": 81.56, "hip": 98.48, "knee": 92.01, "ankle": 85.71}

groups   = list(pdj.per_group.keys())
hrnet_v  = [pdj.per_group[g] * 100 for g in groups]
rtmpose_v = [rtmpose_pdj.get(g, 0) for g in groups]

x, width = np.arange(len(groups)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, hrnet_v,   width, label="HRNet-W48",  color="steelblue")
ax.bar(x + width/2, rtmpose_v, width, label="RTMPose-X",  color="coral")
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=20)
ax.set_ylabel("PDJ@0.5 (%)")
ax.set_title("PDJ@0.5 por grupo anatômico — HRNet vs RTMPose")
ax.set_ylim(0, 105)
ax.legend()
fig.tight_layout()
plt.show()

## MPJPE-2D por joint

In [ ]:
from football_orient_pose.utils.keypoint_mapping import H3WB17_NAMES

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(mpjpe.per_joint[np.newaxis, :], cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(17))
ax.set_xticklabels(H3WB17_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticks([])
ax.set_title("MPJPE-2D por joint (px) — HRNet-W48 zero-shot")
plt.colorbar(im, ax=ax, label="px")
fig.tight_layout()
plt.show()

## Critério para fechar a US #13

- `HRNetEstimator()` instancia e carrega o ONNX sem erro.
- `predict()` retorna `(17, 3)` com confidences positivas em imagem real.
- `predict_h3wb()` retorna `(17, 2)`.
- PDJ@0.5 está dentro de ±5 pp de 56.08% **ou** o desvio está documentado e explicado aqui.

> O HRNet performar significativamente abaixo do RTMPose é o resultado **esperado e desejado**
> para a análise comparativa do trabalho. O erro de quantização dos heatmaps em crops 100×100
> justifica a diferença.